# A2 — JupyterHub: probe ngắn 512×512
Chỉ chạy sau A1 hoặc trên GPU khác. Không có Gradio, không cần Colab/Drive mount. Đây là thử suy luận trên validation, **không phải** lượt fine-tune mới và không chọn tham số bằng test cũ.


In [ ]:
from pathlib import Path
import os, sys, subprocess, json, time
REPO = Path.home()/'SDH_HK261_CO5085'  # Sửa nếu repo được clone chỗ khác.
if not REPO.exists():
    subprocess.run(['git','clone','https://github.com/truong1501/SDH_HK261_CO5085.git',str(REPO)],check=True)
assert (REPO/'A1/src/server.py').is_file(), 'Cần clone/pull bản code mới đã push lên Git'
import torch
print('Python:',sys.version)
assert torch.cuda.is_available(), 'Chọn kernel GPU của công ty; không tự thay CUDA hệ thống'
print('GPU:',torch.cuda.get_device_name(0))
print('VRAM GiB:',torch.cuda.get_device_properties(0).total_memory/2**30)


## Cài phụ thuộc A2
Nếu dùng kernel chung với A1, hoàn tất A1 trước; ưu tiên environment/kernel riêng để tránh thay phiên bản thư viện của tác vụ đang chạy.

In [ ]:
os.chdir(REPO/'A2/instructpix2pix')
result=subprocess.run([sys.executable,'-m','pip','install','-r','requirements.txt'],stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True)
print(result.stdout);result.check_returncode()
DATA=Path.home()/'datasets/ip2p_probe'
if not DATA.exists():subprocess.run([sys.executable,'prepare_data.py','--output',str(DATA)],check=True)


## Probe 512 và guidance
Đầu tiên một cấu hình trên 20 validation ảnh (30 bước suy luận). Nếu còn ngân sách sau A1, bật sweep sáu cấu hình. Mỗi output mới/trống; bỏ qua lượt đã có summary. Không tự train tiếp checkpoint 2.000 cũ. Có thể đặt CHECKPOINT tới folder chứa unet/ nếu đã chuyển trọng số từ Drive.


In [ ]:
CHECKPOINT = None  # Ví dụ REPO/'A2/runs/ip2p_v1/checkpoint-2000'
RUN_SWEEP = False
configs=[(7.5,1.5)] if not RUN_SWEEP else [(t,i) for t in [7.5,10.] for i in [1.2,1.5,1.8]]
OUT=REPO/'A2/runs/probe512'
for text_scale,image_scale in configs:
    output=OUT/f't{text_scale:g}_i{image_scale:g}'
    if (output/'summary.json').exists():
        print((output/'summary.json').read_text());continue
    command=[sys.executable,'compare.py','--data',str(DATA),'--output',str(output),
             '--split','validation','--limit','20','--resolution','512','--steps','30',
             '--text_guidance',str(text_scale),'--image_guidance',str(image_scale)]
    if CHECKPOINT is not None:command += ['--unet',str(CHECKPOINT)]
    subprocess.run(command,check=True)


## Xem kết quả — không cần giao diện riêng

In [ ]:
from IPython.display import display,Image,FileLink
for output in sorted(OUT.glob('*')):
    summary=output/'summary.json'
    if summary.exists():
        print(output.name,summary.read_text())
        display(Image(filename=str(output/'0000_input_target_prediction.png')))


## Bước tiếp theo
Xem `IMPROVEMENT_PLAN.md`. Chưa có loader HQ-Edit hoặc fine-tune 512 mới trong probe này. Dành ngân sách GPU chính cho A1 trước. Giữ nguyên báo cáo run ip2p_v1 làm baseline đã hoàn tất.
